In [1]:
# Load and chunk the dataset from Wikipedia
from langchain_community.document_loaders import WikipediaLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

loader = WikipediaLoader(query="Steve Jobs", load_max_docs=5)
documents = loader.load()

text_splitter = RecursiveCharacterTextSplitter(chunk_size=300, chunk_overlap=100)
docs = text_splitter.split_documents(documents)
print(f"Splitted into {len(docs)} chunk(s)")

c:\code2\Natural Language Processing Projects\RAG System\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Splitted into 108 chunk(s)


In [2]:
# Build vector store 
from langchain_community.vectorstores import FAISS
from langchain_huggingface import HuggingFaceEmbeddings

embedding_model = HuggingFaceEmbeddings(model_name='all-MiniLM-L6-v2')
vector_store = FAISS.from_documents(
    documents=docs,
    embedding=embedding_model
)

In [3]:
# Setup LLM to generate hypothetical answer
import os
from dotenv import load_dotenv
from langchain.chat_models import init_chat_model

load_dotenv()
os.environ['OPENAI_API_KEY'] = os.getenv("OPENAI_API_KEY")

llm = init_chat_model("openai:o4-mini")

In [5]:
from langchain_core.prompts import SystemMessagePromptTemplate, ChatPromptTemplate

def get_hyde_docs(query):
    template = """Imagine you are an expert writing a detailed explanation on the topic: '{query}'
    create a hypothetical answer for the topic"""

    system_prompt = SystemMessagePromptTemplate.from_template(template=template)
    chat_prompt = ChatPromptTemplate.from_messages([system_prompt])
    messages = chat_prompt.format_prompt(query=query).to_messages()
    print(messages)
    response = llm.invoke(messages)
    hypo_doc = response.content
    return hypo_doc

In [6]:
query = "When was Steve Jobs fired from Apple?"
print(get_hyde_docs(query))

[SystemMessage(content="Imagine you are an expert writing a detailed explanation on the topic: 'When was Steve Jobs fired from Apple?'\n    create a hypothetical answer for the topic", additional_kwargs={}, response_metadata={})]
Steve Jobs’s departure from Apple dates back to the summer of 1985, when a protracted board-level power struggle came to head.  Here’s a concise, expert-style breakdown:

1. Background  
   – In 1976, Steve Jobs and Steve Wozniak co-founded Apple Computer in Jobs’s parents’ garage.  
   – By 1983, Apple had grown explosively, and the board brought in former PepsiCo executive John Sculley as CEO to professionalize operations.

2. Rising Tensions  
   – Jobs, then head of the Macintosh division, chafed under Sculley’s more conservative management style.  
   – Early 1985 saw lackluster Macintosh sales and mounting internal disagreements over strategy, marketing and product roadmaps.

3. The Critical Board Meeting (Early September 1985)  
   – In the days leading

In [9]:
retriever = vector_store.as_retriever(
    search_kwargs={"k":5}
)

In [10]:
base_retrieval_docs = retriever.invoke(query)

for doc in base_retrieval_docs:
    print(f"Metadata: {doc.metadata}")
    print(f"Content: {doc.page_content}\n")

Metadata: {'title': 'Steve Jobs', 'summary': 'Steven Paul Jobs (February 24, 1955 – October 5, 2011) was an American businessman, inventor, and investor best known for co-founding the technology company Apple Inc. Jobs was also the founder of NeXT and chairman and majority shareholder of Pixar. He was a pioneer of the personal computer revolution of the 1970s and 1980s, along with his early business partner and fellow Apple co-founder Steve Wozniak.\nJobs was born in San Francisco in 1955 and adopted shortly afterwards. He attended Reed College in 1972 before withdrawing that same year. In 1974, he traveled through India, seeking enlightenment before later studying Zen Buddhism. He and Wozniak co-founded Apple in 1976 to further develop and sell Wozniak\'s Apple I personal computer. Together, the duo gained fame and wealth a year later with production and sale of the Apple II, one of the first highly successful mass-produced microcomputers. \nJobs saw the commercial potential of the Xe

In [12]:
# Using HyDE embedder
from langchain_classic.chains.hyde.base import HypotheticalDocumentEmbedder

hyde_embedding_fn = HypotheticalDocumentEmbedder.from_llm(
    llm=llm,
    base_embeddings=embedding_model,
    prompt_key="web_search"
)

In [ ]:
# Creating vector store using HypotheticalDocumentEmbedder
hyde_vector_store = FAISS.from_documents(
    documents=docs,
    embedding=hyde_embedding_fn
)

In [14]:
hyde_retriever = hyde_vector_store.as_retriever(
    search_kwargs={"k":5}
)

In [15]:
hyde_retrieval_docs = hyde_retriever.invoke(query)

for doc in hyde_retrieval_docs:
    print(f"Metadata: {doc.metadata}")
    print(f"Content: {doc.page_content}\n")

Metadata: {'title': 'Steve Jobs', 'summary': 'Steven Paul Jobs (February 24, 1955 – October 5, 2011) was an American businessman, inventor, and investor best known for co-founding the technology company Apple Inc. Jobs was also the founder of NeXT and chairman and majority shareholder of Pixar. He was a pioneer of the personal computer revolution of the 1970s and 1980s, along with his early business partner and fellow Apple co-founder Steve Wozniak.\nJobs was born in San Francisco in 1955 and adopted shortly afterwards. He attended Reed College in 1972 before withdrawing that same year. In 1974, he traveled through India, seeking enlightenment before later studying Zen Buddhism. He and Wozniak co-founded Apple in 1976 to further develop and sell Wozniak\'s Apple I personal computer. Together, the duo gained fame and wealth a year later with production and sale of the Apple II, one of the first highly successful mass-produced microcomputers. \nJobs saw the commercial potential of the Xe